# GPT2-small CPT v2: SID Semantic Curriculum

Goal: strengthen SID grounding before SFT.

Core changes versus the first smoke CPT:
- every movie participates in metadata grounding in every synthetic epoch;
- descriptions are much rarer and rotate by shards;
- metadata uses many templates, including reverse `metadata -> SID` examples;
- behavior uses fresh train-only sliding windows instead of one static history per user;
- the corpus is materialized as 18 synthetic curriculum epochs, then trained for one pass.

Do not treat this as SFT. This is still CPT: learn the language of SIDs, item semantics, and train-only histories.


## 1. Imports and config

`NUM_SYNTHETIC_EPOCHS` controls how many varied curriculum passes are generated. `NUM_TRAIN_EPOCHS` stays `1.0`, because the generated corpus already contains all synthetic epochs.


In [1]:
from pathlib import Path
from importlib import metadata
from packaging.version import Version
import inspect
import json
import random

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

from transformers import (
    DataCollatorForLanguageModeling,
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    Trainer,
    TrainingArguments,
)


In [2]:
try:
    accelerate_version = metadata.version("accelerate")
except metadata.PackageNotFoundError as exc:
    raise ImportError(
        "This notebook needs accelerate for HuggingFace Trainer. "
        "Install it in this exact kernel with: %pip install -U accelerate"
    ) from exc

if Version(accelerate_version) < Version("0.21.0"):
    raise ImportError(
        f"accelerate>=0.21.0 is required, found {accelerate_version}. "
        "Upgrade in this exact kernel with: %pip install -U accelerate"
    )

print("accelerate:", accelerate_version)


accelerate: 1.13.0


In [3]:
ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

BASE_MODEL = "gpt2"
RUN_NAME = "cpt_gpt2_small_sid_v2_qwen4b_plum_curriculum_v2"

TRAIN_PATH = ROOT / "data/processed/splits/train.parquet"
USERS_PATH = ROOT / "data/raw/ml-1m/users.dat"
ITEM_PROFILE_PATH = ROOT / "data/processed/item_features/qwen4b_audited_v1_item_profiles.parquet"
SID_ARRAY_PATH = ROOT / "runs/qwen4b_rqvae_sid_v2_plum/SIDs_best.npy"
SID_MAPPING_PATH = ROOT / "runs/qwen4b_rqvae_sid_v2_plum/sid_mapping_best.parquet"

ARTIFACT_DIR = ROOT / "data/processed/artifacts" / RUN_NAME
MODEL_OUT_DIR = ARTIFACT_DIR / "model"
FINAL_MODEL_DIR = ARTIFACT_DIR / "final"

SEED = 42
MAX_SEQ_LENGTH = 320
HISTORY_LAST_K = 32
DESC_TEXT_MAX_TOKENS = 64

NUM_SYNTHETIC_EPOCHS = 18  # set 15-20; 18 is the default strengthened CPT run
CORE_META_EXAMPLES_PER_ITEM = 2
DESC_SHARDS = 10  # about 10% of movies get description examples per synthetic epoch
BEHAVIOR_RATIO = 0.50
VALIDATION_SIZE = 0.015
MAX_EVAL_EXAMPLES = 512

NUM_TRAIN_EPOCHS = 1.0
DRY_RUN_STEPS = 0  # set to 1 for a trainer smoke test, then back to 0
LEARNING_RATE = 5e-5
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
PER_DEVICE_BATCH = 8
GRADIENT_ACCUMULATION_STEPS = 4
EVAL_STEPS = 500
SAVE_STEPS = 500
LOGGING_STEPS = 50
SAVE_TOTAL_LIMIT = 2

INCLUDE_RATINGS = True
INCLUDE_USER_FEATURES = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
FP16 = DEVICE == "cuda" and not BF16
PRECISION_MODE = "bf16" if BF16 else ("fp16" if FP16 else "fp32")
if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(ROOT)
print(DEVICE, PRECISION_MODE)


C:\Users\User\plum-ml1m-repro
cuda bf16


## 2. Load tables

Only `train.parquet` is used for behavior. Recommendation `val/test` splits are not used as CPT behavior input.


In [4]:
train = pd.read_parquet(TRAIN_PATH)
users = pd.read_csv(
    USERS_PATH,
    sep="::",
    engine="python",
    names=["user_id", "gender", "age", "occupation", "zip"],
    encoding="latin-1",
)
profiles = pd.read_parquet(ITEM_PROFILE_PATH).sort_values("item_idx").reset_index(drop=True)
sid_mapping = pd.read_parquet(SID_MAPPING_PATH).sort_values("item_idx").reset_index(drop=True)
sids = np.load(SID_ARRAY_PATH)

assert train[["user_id", "user_idx", "item_idx", "rating", "timestamp"]].notna().all().all()
assert np.array_equal(profiles["item_idx"].to_numpy(), np.arange(len(profiles)))
assert np.array_equal(sid_mapping["item_idx"].to_numpy(), np.arange(len(sid_mapping)))
assert sids.shape == (len(profiles), 4)

profiles["title_clean"] = profiles["title_clean"].fillna(profiles["title"].astype(str))
profiles["year_text"] = profiles["year_text"].fillna(profiles["year"].astype(str))
profiles["genres_text"] = profiles["genres_text"].fillna(profiles["genres"].astype(str).str.replace("|", ", ", regex=False))
profiles["description_text"] = profiles["description_text"].fillna("")

print("train rows:", len(train))
print("users:", len(users))
print("items:", len(profiles))
print("sids:", sids.shape)
profiles[["item_idx", "movie_id", "title", "year_text", "genres_text", "description_text"]].head()


train rows: 988129
users: 6040
items: 3706
sids: (3706, 4)


,item_idx,movie_id,title,year_text,genres_text,description_text
0,0,1,Toy Story (1995),1995,"Animation, Children's, Comedy","Plot overview: Led by Woody, Andy's toys live ..."
1,1,2,Jumanji (1995),1995,"Adventure, Children's, Fantasy",Plot overview: When siblings Judy and Peter di...
2,2,3,Grumpier Old Men (1995),1995,"Comedy, Romance",Plot overview: A family wedding reignites the ...
3,3,4,Waiting to Exhale (1995),1995,"Comedy, Drama","Plot overview: Cheated on, mistreated and step..."
4,4,5,Father of the Bride Part II (1995),1995,Comedy,Plot overview: Just when George Banks has reco...


## 3. Tokenizer and schema

Task tokens help the small GPT2 keep metadata, description, and behavior modes separated.


In [5]:
BOS = "<bos>"
EOS = "<eos>"
PAD = "<pad>"
UNK = "<unk>"
USER_OPEN = "<user>"
USER_CLOSE = "</user>"
HIST = "<hist>"
EVENT_OPEN = "<e>"
EVENT_CLOSE = "</e>"
META_OPEN = "<meta>"
META_CLOSE = "</meta>"
DESC_OPEN = "<desc>"
DESC_CLOSE = "</desc>"
TASK_META = "<task_meta>"
TASK_SID = "<task_sid>"
TASK_DESC = "<task_desc>"
TASK_HIST = "<task_hist>"

SCHEMA_TOKENS = [
    USER_OPEN, USER_CLOSE, HIST, EVENT_OPEN, EVENT_CLOSE,
    META_OPEN, META_CLOSE, DESC_OPEN, DESC_CLOSE,
    TASK_META, TASK_SID, TASK_DESC, TASK_HIST,
]


def sid_tokens(sid):
    return [f"<sid_{level}_{int(code)}>" for level, code in enumerate(sid)]


def sid_text(item_idx):
    return " ".join(sid_tokens(sids[int(item_idx)]))


def rating_token(rating):
    return f"<rat_{int(rating)}>"


def user_tokens(row):
    return [f"<gen_{row.gender}>", f"<age_{int(row.age)}>", f"<occ_{int(row.occupation)}>"]


tokenizer = GPT2TokenizerFast.from_pretrained(BASE_MODEL)
tokenizer.add_special_tokens({
    "bos_token": BOS,
    "eos_token": EOS,
    "pad_token": PAD,
    "unk_token": UNK,
})

extra_tokens = set(SCHEMA_TOKENS)
extra_tokens.update(rating_token(r) for r in range(1, 6))
extra_tokens.update(token for sid in sids for token in sid_tokens(sid))

if INCLUDE_USER_FEATURES:
    extra_tokens.update(f"<gen_{v}>" for v in users["gender"].dropna().unique())
    extra_tokens.update(f"<age_{int(v)}>" for v in users["age"].dropna().unique())
    extra_tokens.update(f"<occ_{int(v)}>" for v in users["occupation"].dropna().unique())

n_added = tokenizer.add_tokens(sorted(extra_tokens))
print("vocab size:", len(tokenizer), "new tokens:", n_added)


vocab size: 50642 new tokens: 381


## 4. Template helpers

Each synthetic epoch changes templates and behavior cutoffs. Description examples are sharded, so only a small part of movies contributes plot text per epoch.


In [6]:
CORE_TEMPLATES = [
    "{bos} {task} {meta_open} Movie {sid} has title: {title}. Release year: {year}. Genres: {genres}. {meta_close} {eos}",
    "{bos} {task} {meta_open} Semantic ID {sid} refers to movie: {title}. Year: {year}. Genres: {genres}. {meta_close} {eos}",
    "{bos} {task} {meta_open} Item {sid}. Title: {title}. Year: {year}. Genre labels: {genres}. {meta_close} {eos}",
    "{bos} {task} {meta_open} For {sid}, the movie metadata is title {title}, year {year}, genres {genres}. {meta_close} {eos}",
]

REVERSE_TEMPLATES = [
    "{bos} {task} {meta_open} Movie title: {title}. Release year: {year}. Genres: {genres}. Semantic ID: {sid}. {meta_close} {eos}",
    "{bos} {task} {meta_open} The movie {title} from {year}, with genres {genres}, maps to semantic ID {sid}. {meta_close} {eos}",
    "{bos} {task} {meta_open} Genres: {genres}. Title: {title}. Year: {year}. SID: {sid}. {meta_close} {eos}",
    "{bos} {task} {meta_open} Identify SID for movie {title} ({year}), genres {genres}: {sid}. {meta_close} {eos}",
]

FIELD_TEMPLATES = [
    "{bos} {task} {meta_open} Title of {sid}: {title}. {meta_close} {eos}",
    "{bos} {task} {meta_open} Genres of {sid}: {genres}. {meta_close} {eos}",
    "{bos} {task} {meta_open} Release year of {sid}: {year}. {meta_close} {eos}",
    "{bos} {task} {meta_open} Movie {title} has semantic ID {sid}. {meta_close} {eos}",
    "{bos} {task} {meta_open} Movie with genres {genres} and title {title} uses SID {sid}. {meta_close} {eos}",
]

DESC_TEMPLATES = [
    "{bos} {task} {desc_open} Movie {sid}. Title: {title}. Short plot: {desc} {desc_close} {eos}",
    "{bos} {task} {desc_open} Plot overview for {sid}, {title}: {desc} {desc_close} {eos}",
    "{bos} {task} {desc_open} Title: {title}. Genres: {genres}. Semantic ID: {sid}. Plot: {desc} {desc_close} {eos}",
    "{bos} {task} {desc_open} Plot: {desc} Semantic ID: {sid}. Movie title: {title}. {desc_close} {eos}",
]


def encode_text(text, max_length=MAX_SEQ_LENGTH):
    return tokenizer.encode(text, add_special_tokens=False, truncation=True, max_length=max_length)


def shorten_text(text, max_tokens=DESC_TEXT_MAX_TOKENS):
    ids = tokenizer.encode(str(text), add_special_tokens=False)
    return tokenizer.decode(ids[:max_tokens]).strip()


def fmt(template, row, task):
    return template.format(
        bos=BOS,
        eos=EOS,
        task=task,
        meta_open=META_OPEN,
        meta_close=META_CLOSE,
        desc_open=DESC_OPEN,
        desc_close=DESC_CLOSE,
        sid=sid_text(row.item_idx),
        title=str(row.title_clean).strip(),
        year=str(row.year_text).strip(),
        genres=str(row.genres_text).strip(),
        desc=shorten_text(row.description_text),
    )


def has_description(row):
    text = str(row.description_text).strip()
    return bool(text) and text.lower() != "nan"


## 5. Behavior helpers

For every synthetic epoch, each user gets at least one sampled train-only history window, then extra windows are sampled until behavior count matches metadata count.


In [7]:
users_by_id = users.set_index("user_id", drop=False)
train_sorted = train.sort_values(["user_idx", "timestamp"]).reset_index(drop=True)
user_events = {
    user_id: group[["item_idx", "rating", "timestamp"]].to_dict("records")
    for user_id, group in train_sorted.groupby("user_id", sort=False)
}
user_ids = np.array(list(user_events.keys()))


def behavior_event(event):
    tokens = [EVENT_OPEN]
    tokens.extend(sid_tokens(sids[int(event["item_idx"])]))
    if INCLUDE_RATINGS:
        tokens.append(rating_token(event["rating"]))
    tokens.append(EVENT_CLOSE)
    return tokens


def fit_behavior_tokens(prefix, events):
    selected = list(events[-HISTORY_LAST_K:])
    while selected:
        tokens = prefix + [tok for event in selected for tok in event] + [EOS]
        ids = tokenizer.convert_tokens_to_ids(tokens)
        if len(ids) <= MAX_SEQ_LENGTH:
            return ids
        selected = selected[1:]
    return tokenizer.convert_tokens_to_ids(prefix + [EOS])


def make_behavior_example(user_id, rng):
    events = user_events[user_id]
    if len(events) <= 1:
        cutoff = len(events)
    else:
        cutoff = int(rng.integers(1, len(events) + 1))
    window = events[:cutoff]

    prefix = [BOS, TASK_HIST]
    if INCLUDE_USER_FEATURES and user_id in users_by_id.index:
        prefix.extend([USER_OPEN, *user_tokens(users_by_id.loc[user_id]), USER_CLOSE])
    prefix.append(HIST)

    return fit_behavior_tokens(prefix, [behavior_event(event) for event in window])


## 6. Build curriculum corpus

This is the expensive data-preparation cell, but it does not train the model. Check the printed counts before training.

The eval set is only a monitor sample from the generated CPT curriculum. It does not remove examples from training, because the goal here is maximal SID grounding coverage, not a strict held-out language-model benchmark.


In [8]:
rows = []
epoch_stats = []
profile_rows = list(profiles.itertuples(index=False))

for epoch in range(NUM_SYNTHETIC_EPOCHS):
    rng = np.random.default_rng(SEED + epoch)
    epoch_rows = []

    for row in profile_rows:
        template_plan = [CORE_TEMPLATES[int(rng.integers(0, len(CORE_TEMPLATES)))]]
        extra_pool = REVERSE_TEMPLATES + FIELD_TEMPLATES
        while len(template_plan) < CORE_META_EXAMPLES_PER_ITEM:
            template_plan.append(extra_pool[int(rng.integers(0, len(extra_pool)))])

        for j, template in enumerate(template_plan):
            task = TASK_META if j == 0 else TASK_SID
            source = "metadata_core" if j == 0 else "metadata_reverse_or_field"
            epoch_rows.append({
                "input_ids": encode_text(fmt(template, row, task)),
                "source": source,
                "synthetic_epoch": epoch,
                "item_idx": int(row.item_idx),
            })

    desc_candidates = [
        row for row in profile_rows
        if has_description(row) and int(row.item_idx) % DESC_SHARDS == epoch % DESC_SHARDS
    ]
    rng.shuffle(desc_candidates)
    for row in desc_candidates:
        desc_template = DESC_TEMPLATES[int(rng.integers(0, len(DESC_TEMPLATES)))]
        epoch_rows.append({
            "input_ids": encode_text(fmt(desc_template, row, TASK_DESC)),
            "source": "metadata_description",
            "synthetic_epoch": epoch,
            "item_idx": int(row.item_idx),
        })

    n_metadata = len(epoch_rows)
    n_behavior = int(round(n_metadata * BEHAVIOR_RATIO / (1.0 - BEHAVIOR_RATIO)))

    selected_users = list(user_ids)
    if n_behavior > len(selected_users):
        extra = rng.choice(user_ids, size=n_behavior - len(selected_users), replace=True).tolist()
        selected_users.extend(extra)
    else:
        rng.shuffle(selected_users)
        selected_users = selected_users[:n_behavior]

    for user_id in selected_users:
        epoch_rows.append({
            "input_ids": make_behavior_example(int(user_id), rng),
            "source": "behavior_window",
            "synthetic_epoch": epoch,
            "item_idx": -1,
        })

    rng.shuffle(epoch_rows)
    rows.extend(epoch_rows)
    epoch_stats.append(pd.Series([r["source"] for r in epoch_rows]).value_counts().to_dict())

rng = np.random.default_rng(SEED)
rng.shuffle(rows)

val_n = min(MAX_EVAL_EXAMPLES, max(1, int(len(rows) * VALIDATION_SIZE)))
val_idx = rng.choice(len(rows), size=val_n, replace=False)
val_rows = [rows[int(i)] for i in val_idx]
train_rows = rows  # keep every generated curriculum example in training

source_counts = pd.Series([r["source"] for r in rows]).value_counts().to_dict()
lengths = np.array([len(r["input_ids"]) for r in rows], dtype=np.int32)

curriculum_stats = {
    "run_name": RUN_NAME,
    "total_examples": len(rows),
    "train_examples": len(train_rows),
    "validation_examples": len(val_rows),
    "validation_is_monitor_sample_from_curriculum": True,
    "num_synthetic_epochs": NUM_SYNTHETIC_EPOCHS,
    "items": len(profiles),
    "all_items_in_core_metadata_each_epoch": True,
    "core_meta_examples_per_item_per_epoch": CORE_META_EXAMPLES_PER_ITEM,
    "desc_shards": DESC_SHARDS,
    "behavior_ratio_target": BEHAVIOR_RATIO,
    "source_counts": source_counts,
    "length_min": int(lengths.min()),
    "length_p50": int(np.percentile(lengths, 50)),
    "length_p95": int(np.percentile(lengths, 95)),
    "length_max": int(lengths.max()),
    "max_seq_length": MAX_SEQ_LENGTH,
    "history_last_k": HISTORY_LAST_K,
    "desc_text_max_tokens": DESC_TEXT_MAX_TOKENS,
}

print(json.dumps(curriculum_stats, indent=2))
pd.DataFrame(epoch_stats).fillna(0).astype(int).head()


{
  "run_name": "cpt_gpt2_small_sid_v2_qwen4b_plum_curriculum_v2",
  "total_examples": 280176,
  "train_examples": 280176,
  "validation_examples": 512,
  "validation_is_monitor_sample_from_curriculum": true,
  "num_synthetic_epochs": 18,
  "items": 3706,
  "all_items_in_core_metadata_each_epoch": true,
  "core_meta_examples_per_item_per_epoch": 2,
  "desc_shards": 10,
  "behavior_ratio_target": 0.5,
  "source_counts": {
    "behavior_window": 140088,
    "metadata_reverse_or_field": 66708,
    "metadata_core": 66708,
    "metadata_description": 6672
  },
  "length_min": 16,
  "length_p50": 46,
  "length_p95": 233,
  "length_max": 233,
  "max_seq_length": 320,
  "history_last_k": 32,
  "desc_text_max_tokens": 64
}


,behavior_window,metadata_reverse_or_field,metadata_core,metadata_description
0,7783,3706,3706,371
1,7783,3706,3706,371
2,7783,3706,3706,371
3,7783,3706,3706,371
4,7783,3706,3706,371


In [9]:
preview = []
for source in ["metadata_core", "metadata_reverse_or_field", "metadata_description", "behavior_window"]:
    row = next(r for r in rows if r["source"] == source)
    preview.append({
        "source": source,
        "synthetic_epoch": row["synthetic_epoch"],
        "text": tokenizer.decode(row["input_ids"]),
    })

pd.DataFrame(preview)


,source,synthetic_epoch,text
0,metadata_core,13,<bos> <task_meta> <meta> Movie <sid_0_461> <si...
1,metadata_reverse_or_field,1,<bos> <task_sid> <meta> Movie Dadetown has sem...
2,metadata_description,13,<bos> <task_desc> <desc> Plot: Plot overview: ...
3,behavior_window,6,<bos><task_hist><user><gen_M><age_25><occ_0></...


## 7. Save curriculum artifacts

The full corpus is saved so the training cell can be restarted without rebuilding examples.


In [10]:
tokenizer.save_pretrained(ARTIFACT_DIR / "tokenizer")
np.save(ARTIFACT_DIR / "SIDs_best.npy", sids)

torch.save(
    {
        "train_rows": train_rows,
        "val_rows": val_rows,
        "curriculum_stats": curriculum_stats,
    },
    ARTIFACT_DIR / "cpt_curriculum_corpus.pt",
)

manifest = {
    "run_name": RUN_NAME,
    "base_model": BASE_MODEL,
    "train_path": str(TRAIN_PATH),
    "item_profile_path": str(ITEM_PROFILE_PATH),
    "sid_array_path": str(SID_ARRAY_PATH),
    "sid_mapping_path": str(SID_MAPPING_PATH),
    "uses_recsys_val_test_as_behavior": False,
    "curriculum_stats": curriculum_stats,
}
with (ARTIFACT_DIR / "manifest.json").open("w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print(ARTIFACT_DIR)


C:\Users\User\plum-ml1m-repro\data\processed\artifacts\cpt_gpt2_small_sid_v2_qwen4b_plum_curriculum_v2


## 8. Train GPT2-small CPT v2

Set `DRY_RUN_STEPS = 1` first if you want a one-step smoke test. For the real run, keep `DRY_RUN_STEPS = 0`.


In [11]:
class LMListDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return {"input_ids": torch.tensor(self.rows[idx]["input_ids"], dtype=torch.long)}


payload = torch.load(ARTIFACT_DIR / "cpt_curriculum_corpus.pt", weights_only=False)
train_ds = LMListDataset(payload["train_rows"])
val_ds = LMListDataset(payload["val_rows"])

tokenizer = GPT2TokenizerFast.from_pretrained(ARTIFACT_DIR / "tokenizer")
model = GPT2LMHeadModel.from_pretrained(BASE_MODEL)
model.resize_token_embeddings(len(tokenizer))
model.config.bos_token_id = tokenizer.bos_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, pad_to_multiple_of=8)

args_kwargs = {
    "output_dir": str(MODEL_OUT_DIR),
    "overwrite_output_dir": True,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "max_steps": DRY_RUN_STEPS if DRY_RUN_STEPS > 0 else -1,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "per_device_train_batch_size": PER_DEVICE_BATCH,
    "per_device_eval_batch_size": PER_DEVICE_BATCH,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "logging_steps": LOGGING_STEPS,
    "eval_steps": EVAL_STEPS,
    "save_steps": SAVE_STEPS,
    "save_strategy": "steps",
    "save_total_limit": SAVE_TOTAL_LIMIT,
    "fp16": FP16,
    "bf16": BF16,
    "report_to": "none",
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "dataloader_num_workers": 0,
    "dataloader_pin_memory": DEVICE == "cuda",
}

args_params = inspect.signature(TrainingArguments).parameters
if "eval_strategy" in args_params:
    args_kwargs["eval_strategy"] = "steps"
else:
    args_kwargs["evaluation_strategy"] = "steps"

training_args = TrainingArguments(**{k: v for k, v in args_kwargs.items() if k in args_params})
if DRY_RUN_STEPS > 0:
    print(f"DRY RUN MODE: training will stop after {DRY_RUN_STEPS} step(s).")

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_ds,
    "eval_dataset": val_ds,
    "data_collator": collator,
}
trainer_params = inspect.signature(Trainer).parameters
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

print("precision:", PRECISION_MODE, "bf16=", BF16, "fp16=", FP16)
print("train examples:", len(train_ds), "val examples:", len(val_ds))
trainer = Trainer(**trainer_kwargs)
trainer


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


precision: bf16 bf16= True fp16= False
train examples: 280176 val examples: 512


In [12]:
train_output = trainer.train()

if DRY_RUN_STEPS > 0:
    result = {
        "dry_run": True,
        "train_output": train_output.metrics,
        "curriculum_stats": payload["curriculum_stats"],
        "note": "Dry run finished. Set DRY_RUN_STEPS=0 and rerun training for the full CPT checkpoint.",
    }
else:
    trainer.save_model(str(FINAL_MODEL_DIR))
    tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

    result = {
        "dry_run": False,
        "train_output": train_output.metrics,
        "curriculum_stats": payload["curriculum_stats"],
        "final_model_dir": str(FINAL_MODEL_DIR),
    }
    with (ARTIFACT_DIR / "train_result.json").open("w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

result


Step,Training Loss,Validation Loss
500,2.185800,2.045595
1000,1.655600,1.521406
1500,1.395300,1.295162
2000,1.282200,1.203253
2500,1.216900,1.147897
3000,1.176900,1.115692
3500,1.143600,1.084049
4000,1.134100,1.068669
4500,1.128600,1.053369
5000,1.088100,1.045296


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


{'dry_run': False,
 'train_output': {'train_runtime': 2905.4309,
  'train_samples_per_second': 96.432,
  'train_steps_per_second': 3.013,
  'total_flos': 3.3338021732352e+16,
  'train_loss': 1.3808121299416865,
  'epoch': 0.9999428930386615},
 'curriculum_stats': {'run_name': 'cpt_gpt2_small_sid_v2_qwen4b_plum_curriculum_v2',
  'total_examples': 280176,
  'train_examples': 280176,
  'validation_examples': 512,
  'validation_is_monitor_sample_from_curriculum': True,
  'num_synthetic_epochs': 18,
  'items': 3706,
  'all_items_in_core_metadata_each_epoch': True,
  'core_meta_examples_per_item_per_epoch': 2,
  'desc_shards': 10,
  'behavior_ratio_target': 0.5,
  'source_counts': {'behavior_window': 140088,
   'metadata_reverse_or_field': 66708,
   'metadata_core': 66708,
   'metadata_description': 6672},
  'length_min': 16,
  'length_p50': 46,
  'length_p95': 233,
  'length_max': 233,
  'max_seq_length': 320,
  'history_last_k': 32,
  'desc_text_max_tokens': 64},
 'final_model_dir': 'C:\\U

## 9. Quick generation sanity check after training

Run this only after the full CPT v2 checkpoint is saved.


In [13]:
MODEL_DIR = FINAL_MODEL_DIR

tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_DIR)
model = GPT2LMHeadModel.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.bfloat16 if BF16 else (torch.float16 if FP16 else torch.float32),
).to(DEVICE)
model.eval()


def generate(prompt, max_new_tokens=80, do_sample=False, temperature=0.8, top_p=0.9):
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_p=top_p if do_sample else None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.08,
        )
    return tokenizer.decode(out[0], skip_special_tokens=False)

for item_idx in [0, 1, 1104, 1178, 1192]:
    row = profiles.iloc[item_idx]
    prompt = f"{BOS} {TASK_META} {META_OPEN} Movie {sid_text(item_idx)} has title:"
    print("=" * 100)
    print("TRUE:", row["title"], "|", row["year_text"], "|", row["genres_text"])
    print(generate(prompt, max_new_tokens=70, do_sample=False))


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


TRUE: Toy Story (1995) | 1995 | Animation, Children's, Comedy
<bos> <task_meta> <meta> Movie <sid_0_32> <sid_1_134> <sid_2_20> <sid_3_18> has title: Big Combo, The. Release year: 1997. Genres: Comedy, Drama. </meta> <eos>
TRUE: Jumanji (1995) | 1995 | Adventure, Children's, Fantasy
<bos> <task_meta> <meta> Movie <sid_0_469> <sid_1_18> <sid_2_69> <sid_3_31> has title: Little Princess, The. Release year: 1997. Genres: Adventure, Children's, Fantasy. </meta> <eos>
TRUE: One Flew Over the Cuckoo's Nest (1975) | 1975 | Drama
<bos> <task_meta> <meta> Movie <sid_0_227> <sid_1_113> <sid_2_59> <sid_3_30> has title: Man of No Importance, A. Release year: 1962. Genres: Drama. </meta> <eos>
TRUE: Back to the Future (1985) | 1985 | Comedy, Sci-Fi
<bos> <task_meta> <meta> Movie <sid_0_148> <sid_1_28> <sid_2_66> <sid_3_3> has title: Man Who Knew Too Little, The. Release year: 1984. Genres: Comedy, Drama. </meta> <eos>
TRUE: Big Sleep, The (1946) | 1946 | Film-Noir, Mystery
<bos> <task_meta> <meta> Mo

In [14]:
def event_text(item_idx, rating=None):
    text = f"{EVENT_OPEN} {sid_text(item_idx)}"
    if rating is not None:
        text += f" {rating_token(rating)}"
    return text + f" {EVENT_CLOSE}"


def make_history_prompt(user_id, last_k=8, force_new_event=True):
    user_row = users_by_id.loc[user_id]
    hist = train[train["user_id"] == user_id].sort_values("timestamp").tail(last_k)
    events = " ".join(event_text(row.item_idx, row.rating) for row in hist.itertuples(index=False))
    prompt = f"{BOS} {TASK_HIST} {USER_OPEN} {' '.join(user_tokens(user_row))} {USER_CLOSE} {HIST} {events}"
    if force_new_event:
        prompt += f" {EVENT_OPEN}"
    return prompt

prompt = make_history_prompt(user_id=1, last_k=8, force_new_event=True)
print(generate(prompt, max_new_tokens=40, do_sample=False))


<bos> <task_hist> <user> <gen_F> <age_1> <occ_10> </user> <hist> <e> <sid_0_469> <sid_1_139> <sid_2_93> <sid_3_23> <rat_3> </e> <e> <sid_0_32> <sid_1_134> <sid_2_109> <sid_3_52> <rat_3> </e> <e> <sid_0_469> <sid_1_77> <sid_2_84> <sid_3_33> <rat_4> </e> <e> <sid_0_32> <sid_1_134> <sid_2_20> <sid_3_18> <rat_5> </e> <e> <sid_0_469> <sid_1_138> <sid_2_89> <sid_3_48> <rat_4> </e> <e> <sid_0_32> <sid_1_216> <sid_2_30> <sid_3_26> <rat_5> </e> <e> <sid_0_469> <sid_1_7> <sid_2_49> <sid_3_26> <rat_4> </e> <e> <sid_0_469> <sid_1_102> <sid_2_41> <sid_3_49> <rat_4> </e> <e><sid_0_469> <sid_2_83>. </meta> <eos>


## 10. Expected signal

Compared with the smoke CPT, the first sanity check should improve in two ways:
- metadata prompts should no longer collapse to the same fake title for many SIDs;
- history prompts with forced `<e>` should continue with SID tokens rather than jumping into title/description prose.
